In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [4]:
X = df.drop('Outcome',axis=1).values
y = df['Outcome'].values

In [5]:
X.shape

(768, 8)

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [7]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Dropout

In [9]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])

c:\Users\nasrullah\.conda\envs\tf_env\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
history = model.fit(X_train,y_train,epochs=100,batch_size=32,validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.5522 - loss: 0.7218 - val_accuracy: 0.6688 - val_loss: 0.5980
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6369 - loss: 0.6405 - val_accuracy: 0.7078 - val_loss: 0.5648
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6852 - loss: 0.5976 - val_accuracy: 0.7338 - val_loss: 0.5449
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7165 - loss: 0.5777 - val_accuracy: 0.7532 - val_loss: 0.5305
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7428 - loss: 0.5280 - val_accuracy: 0.7338 - val_loss: 0.5207
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7574 - loss: 0.5309 - val_accuracy: 0.7403 - val_loss: 0.5132
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7652 - loss: 0.5093 - val_accuracy: 0.7338 - val_loss: 0.5083
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7597 - loss: 0.5112 - val_accuracy: 0.7532 

1. How To Select Appropriate Optimizer
2. Number of Nodes in a layer
3. Number of layers in a model
4. All in all one model

In [11]:
# !pip install keras-tuner --upgrade

In [12]:
import keras_tuner as kt

## 1. How To Select Appropriate Optimizer

In [13]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(32, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    optimizer = hp.Choice('optimizer' , values = ['SGD','RMSprop','Adagrad','Adam'])

    model.compile(optimizer = optimizer,loss = 'binary_crossentropy',metrics = ['accuracy'])

    return model

In [14]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials = 5,
                        project_name = 'Optimizer_Select')

Reloading Tuner from .\Optimizer_Select\tuner0.json


In [15]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [16]:
# Get Best Optimizer
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'Adam'}

In [17]:
# Get Best Model
model = tuner.get_best_models(num_models=1)[0]

c:\Users\nasrullah\.conda\envs\tf_env\Lib\site-packages\keras\src\saving\saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
history = model.fit(X_train,y_train,epochs=100,initial_epoch=5,batch_size=32,validation_data=(X_test,y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.7385 - loss: 0.5111 - val_accuracy: 0.7597 - val_loss: 0.5038
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7477 - loss: 0.4984 - val_accuracy: 0.7792 - val_loss: 0.4956
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7502 - loss: 0.4816 - val_accuracy: 0.7857 - val_loss: 0.4931
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7539 - loss: 0.4951 - val_accuracy: 0.7857 - val_loss: 0.4924
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7742 - loss: 0.4565 - val_accuracy: 0.7792 - val_loss: 0.4898
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7761 - loss: 0.4657 - val_accuracy: 0.7792 - val_loss: 0.4889
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7460 - loss: 0.4759 - val_accuracy: 0.7857 - val_loss: 0.4884
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7599 - loss: 0.4653 - val_accuracy: 0.779

## 2. How to Select Number of Nodes in a layer

In [20]:
def build_model(hp):
    model = Sequential()
    units = hp.Int('units', min_value=8, max_value=128)
    model.add(Dense(units=units, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='RMSprop', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [21]:
tuner1 = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials = 5,
                        project_name = 'num_neurons'
                        )

Reloading Tuner from .\num_neurons\tuner0.json


In [22]:
tuner1.search(X_train,y_train,epochs = 5,validation_data = (X_test,y_test))

In [23]:
tuner1.get_best_hyperparameters()[0].values

{'num_layers': 7}

In [24]:
best_hp = tuner1.get_best_hyperparameters()[0]
model = tuner1.hypermodel.build(best_hp)

In [25]:
model.fit(X_train,y_train,epochs=100,initial_epoch=5,batch_size=32,validation_data=(X_test,y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.6489 - loss: 0.7416 - val_accuracy: 0.6429 - val_loss: 0.7401
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6538 - loss: 0.7030 - val_accuracy: 0.6429 - val_loss: 0.7111
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6620 - loss: 0.6762 - val_accuracy: 0.6558 - val_loss: 0.6857
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6434 - loss: 0.6649 - val_accuracy: 0.6688 - val_loss: 0.6616
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6851 - loss: 0.6231 - val_accuracy: 0.6623 - val_loss: 0.6382
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6977 - loss: 0.6096 - val_accuracy: 0.6753 - val_loss: 0.6218
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6677 - loss: 0.6179 - val_accuracy: 0.6818 - val_loss: 0.6054
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6838 - loss: 0.5893 - val_accuracy: 0.720

## 3. How To Select Number of layers in a model

In [26]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(117,activation='relu',input_dim=8))

    for i in range(hp.Int('num_layers', min_value = 1 , max_value = 10)):
        model.add(Dense(117,activation='relu'))
    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='RMSprop', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [27]:
tuner2 = kt.RandomSearch(build_model,
                         objective = 'val_accuracy',
                         max_trials = 5,
                         project_name = 'num_layers')

Reloading Tuner from .\num_layers\tuner0.json


In [28]:
tuner2.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [29]:
tuner2.get_best_hyperparameters()[0].values

{'num_layers': 10}

In [30]:
model = tuner2.get_best_models(num_models=1)[0]

c:\Users\nasrullah\.conda\envs\tf_env\Lib\site-packages\keras\src\saving\saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [31]:
model.fit(X_train,y_train,epochs=100,initial_epoch=5,batch_size=32,validation_data=(X_test,y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.7801 - loss: 0.4641 - val_accuracy: 0.7597 - val_loss: 0.5645
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7772 - loss: 0.4524 - val_accuracy: 0.7403 - val_loss: 0.5863
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8064 - loss: 0.3982 - val_accuracy: 0.7727 - val_loss: 0.6162
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7809 - loss: 0.4367 - val_accuracy: 0.7597 - val_loss: 0.5168
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8202 - loss: 0.4281 - val_accuracy: 0.7727 - val_loss: 0.6871
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8095 - loss: 0.4090 - val_accuracy: 0.7597 - val_loss: 0.5637
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8378 - loss: 0.3437 - val_accuracy: 0.7403 - val_loss: 0.6182
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8448 - loss: 0.3977 - val_accuracy: 

## 4. All in One model

In [51]:
def build_model(hp):
    model = Sequential()
    counter = 0

    for i in range(hp.Int('num_layers',min_value=1, max_value=10)):
        if counter == 0:
            model.add(
                Dense(
                    hp.Int('units' + str(i),min_value=8 , max_value=128 , step=8),
                    activation = hp.Choice('activation' + str(i) , values = ['relu','tanh','sigmoid']),
                    input_dim = 8
                )
            )
            model.add(Dropout(hp.Choice('dropout' + str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        
        else:
            model.add(
                Dense(
                    hp.Int('units' + str(i) , min_value = 8 , max_value = 128 , step=8),
                    activation = hp.Choice('activation' + str(i) , values = ['relu','tanh','sigmoid'])
                )
            )
            model.add(Dropout(hp.Choice('dropout' + str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        counter +=1
        model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer=hp.Choice('optimizers',values = ['Adam','RMSprop','SGD','Nadam','Adadelta']),
                  loss = 'binary_crossentropy',
                  metrics = ['accuracy']
                  )

    return model


In [52]:
tuner3 = kt.RandomSearch(build_model,
                         objective = 'val_accuracy',
                         max_trials=5,
                         project_name = 'All_in_One')

Reloading Tuner from .\All_in_One\tuner0.json


In [53]:
tuner3.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [ ]:
tuner3.get_best_hyperparameters()[0].values

{'num_layers': 1,
 'units0': 24,
 'activation0': 'relu',
 'optimizers': 'RMSprop',
 'units1': 56,
 'activation1': 'tanh',
 'units2': 16,
 'activation2': 'relu',
 'units3': 48,
 'activation3': 'tanh',
 'units4': 24,
 'activation4': 'sigmoid',
 'units5': 80,
 'activation5': 'relu',
 'units6': 104,
 'activation6': 'tanh'}

In [55]:
model = tuner3.get_best_models(num_models=1)[0]

In [56]:
model.fit(X_train,y_train,epochs=100,initial_epoch=5,validation_data=(X_test,y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.6761 - loss: 0.5968 - val_accuracy: 0.7468 - val_loss: 0.5620
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7052 - loss: 0.5604 - val_accuracy: 0.7532 - val_loss: 0.5457
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7338 - loss: 0.5350 - val_accuracy: 0.7597 - val_loss: 0.5328
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7419 - loss: 0.5260 - val_accuracy: 0.7662 - val_loss: 0.5231
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7470 - loss: 0.5182 - val_accuracy: 0.7532 - val_loss: 0.5153
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7584 - loss: 0.4872 - val_accuracy: 0.7792 - val_loss: 0.5078
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7557 - loss: 0.4927 - val_accuracy: 0.7727 - val_loss: 0.5029
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7630 - loss: 0.4822 - val_accuracy: 0